[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ddmms/camml-tutorials/blob/main/notebooks/05-generative/chemeleon-crystals.ipynb)


# Chemeleon-DNG in Practice: DNG and CSP for crystal generation

<img src="https://raw.githubusercontent.com/hspark1212/chemeleon/main/assets/logo_static.jpg" alt="Chemeleon logo figure from the official repository" width="52%"> <img src="https://raw.githubusercontent.com/hspark1212/chemeleon/main/assets/trajectory.gif" alt="Chemeleon trajectory gif from the official repository" width="18%">

*Source:* official Chemeleon repository assets `assets/logo_static.jpg` and `assets/trajectory.gif`: <https://github.com/hspark1212/chemeleon>

The main Chemeleon repository is text-guided, while this notebook focuses on the `chemeleon-dng` branch for DNG and CSP. These upstream visuals still help because they show the broader project identity and a real crystal-generation trajectory from the same model family.

This notebook follows the same scientific pattern as the MatterGen notebook, but the user-facing interface is different. Chemeleon-DNG exposes two explicit tasks: open-ended **DNG** and formula-conditioned **CSP**.

## Where this fits in the course

- `diffusion-fundamentals.ipynb`: the forward / reverse diffusion story is unchanged.
- `crystal-diffusion-from-scratch.ipynb`: the crystal representation and screening logic should already feel familiar.
- `mattergen-crystals.ipynb`: use that notebook as the reference point for scalar-property steering, then use this notebook to contrast task-oriented control.

## Aims

- install and run Chemeleon-DNG in a notebook-friendly way,
- generate crystals in open-ended DNG mode,
- steer DNG with different atom-count schedules,
- generate formula-conditioned candidates with CSP,
- compare conditioning and screening in a second modern crystal-generation workflow.

## Learning outcomes

By the end you should be able to:

1. explain the difference between DNG and CSP,
2. identify which Chemeleon controls act *during* sampling and which diagnostics act *after* sampling,
3. compare formula-conditioned generation to MatterGen's scalar-target workflow,
4. decide when a task-oriented toolkit is the better fit for a computational-chemistry question.

## How to use this notebook

1. Run the DNG quickstart first so the environment and checkpoints are in place.
2. Treat the DNG steering study as a lesson in **sampling control**, not in explicit conditioning.
3. Treat the CSP study as the main conditional-generation demo because the requested formula is directly verifiable.
4. Keep one running note with three labels: `task`, `control knob`, `screening statistic`.


## Primary sources and upstream code

This notebook is organized around the official Chemeleon resources:

- Chemeleon repository and paper: https://github.com/hspark1212/chemeleon and https://www.nature.com/articles/s41467-025-59636-y
- Chemeleon-DNG repository: https://github.com/hspark1212/chemeleon-dng
- Chemeleon-DNG commit pinned in this notebook: `0d8da3a82a0c2211245a1b1394b599ca0545883c`

The goal is not to retrain the model family. The goal is to understand how to run, inspect, and compare the two user-facing tasks in practice.


## Table of Contents

1. [Setup](#1-setup)
2. [How it works](#2-how-it-works)
3. [DNG quickstart](#3-dng-quickstart-generate-crystals-from-scratch)
4. [DNG steering and output analysis](#4-dng-steering-and-output-analysis)
5. [CSP quickstart: formula-conditioned generation](#5-csp-quickstart-formula-conditioned-generation)
6. [CSP distributions, galleries, and screening](#6-csp-distributions-galleries-and-screening)
7. [When to choose Chemeleon-DNG](#7-when-to-choose-chemeleon-dng)
8. [Exercises](#8-exercises)
9. [Troubleshooting](#9-troubleshooting)
10. [Next steps](#10-next-steps)


**Task for you**
<div class="alert alert-block alert-info">

- Before you run anything, write one sentence that distinguishes **open-ended generation**, **sampling control**, and **explicit conditioning**.
- Keep comparing to `mattergen-crystals.ipynb`: what is the first question a user asks in each toolkit?
- Decide ahead of time which Chemeleon task seems closer to the inverse-design problems you care about.

</div>


## 1) Setup

This section installs a clean `chemeleon-dng` environment and leaves the checkpoint download to the first real sampling call.

A small but important implementation detail: the notebook creates the Python environment in the system temporary directory rather than inside the repo. That avoids incomplete `torch` and `numpy` installs on mounted drives such as OneDrive, WSL bind mounts, and some Colab-backed filesystems.

What this setup cell does:

- validates or reclones the `chemeleon-dng` repository,
- checks out the pinned commit `0d8da3a82a0c`,
- creates a clean Python 3.11 environment in a temporary directory,
- installs `chemeleon-dng` into that environment,
- verifies that `chemeleon_dng`, `numpy`, and `torch` all import cleanly,
- defines the helper functions used by the DNG and CSP demos below.

The first DNG or CSP sample will automatically download about $523\,\mathrm{{MB}}$ of pretrained checkpoints into `chemeleon_dng_repo/ckpts/`. Later runs reuse those files.


In [ ]:
from pathlib import Path
import html
import sys


def ensure_day5_helpers_on_path():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        for helper_dir in (
            candidate / "gen_helpers",
            candidate / "notebooks" / "05-generative" / "gen_helpers",
        ):
            if helper_dir.exists():
                parent = helper_dir.parent
                if str(parent) not in sys.path:
                    sys.path.insert(0, str(parent))
                return parent
    raise FileNotFoundError("Could not locate notebooks/05-generative/gen_helpers")


GEN_HELPERS_ROOT = ensure_day5_helpers_on_path()

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from IPython.display import display

from gen_helpers.chemeleon_helpers import (
    analyze_csp_samples,
    analyze_dng_runs,
    sample_chemeleon_dng as _sample_chemeleon_dng,
    setup_chemeleon_dng_environment,
    show_atoms_gallery,
)

chemeleon_env = setup_chemeleon_dng_environment()
notebook_root = chemeleon_env["notebook_root"]
CHEMELEON_DNG_REPO = chemeleon_env["repo_dir"]
CHEMELEON_DNG_OUT = chemeleon_env["output_dir"]
CHEMELEON_DNG_VENV = chemeleon_env["venv_dir"]
CHEMELEON_DNG_PYTHON = chemeleon_env["python_path"]
CHEMELEON_DNG_CLI = chemeleon_env["cli_path"]
CHEMELEON_DNG_DEVICE = chemeleon_env["device"]
CHEMELEON_DNG_DEMO_TIMESTEPS = chemeleon_env["demo_timesteps"]
CHEMELEON_DNG_COMMIT = chemeleon_env["pinned_commit"]
CHEMELEON_DNG_CHECKPOINT_DATASET = 'alex_mp_20'
CHEMELEON_DNG_PRIOR_NAME = 'mp-20'
CHEMELEON_DNG_ALEX_MP20_MODEL_PATH = 'ckpts/chemeleon_dng_alex_mp_20_v0.0.2.ckpt'
CHEMELEON_CSP_ALEX_MP20_MODEL_PATH = 'ckpts/chemeleon_csp_alex_mp_20_v0.0.2.ckpt'


def sample_chemeleon_dng(**kwargs):
    return _sample_chemeleon_dng(chemeleon_env, **kwargs)


def format_widget_pre(text: str) -> str:
    return f"<pre style='white-space:pre-wrap; margin:0'>{html.escape(text)}</pre>"


def bind_widget_state(controls, apply_fn):
    state_holder = {"has_rendered": False, "last": None}

    def refresh(change=None):
        state = {name: control.value for name, control in controls.items()}
        state_key = tuple((name, repr(value)) for name, value in state.items())
        if state_holder["has_rendered"] and state_holder["last"] == state_key:
            return
        state_holder["has_rendered"] = True
        state_holder["last"] = state_key
        apply_fn(**state)

    refresh()
    for control in controls.values():
        control.observe(refresh, names='value')
    return refresh


## 2) How it works

Chemeleon-DNG exposes one diffusion backbone through two user-facing tasks: DNG and CSP.

### Shared backbone
At the model level, both tasks follow the same core recipe:

1. represent a crystal by atom identities, fractional coordinates, and lattice information,
2. corrupt that representation step by step,
3. train a denoiser to reverse the corruption,
4. sample by iteratively denoising from a simple starting distribution.

### Where the control enters

- **DNG:** no target formula is provided. The main control shown here is the atom-count distribution, which biases the search toward smaller or larger cells.
- **CSP:** the target formula is fixed before sampling. The model then searches for plausible structures consistent with that chemistry.

### Comparison with MatterGen

- MatterGen keeps the same raw-crystal viewpoint but adds scalar-property steering.
- Chemeleon-DNG keeps the diffusion backbone but reorganizes the workflow around task choice.
- That means the first user question changes:
  - **MatterGen:** what scalar target do I want?
  - **Chemeleon-DNG:** do I want open-ended discovery or fixed-formula search?

> **Quick check:** what is shared between DNG and CSP, and what changes?
>
><details><summary>Answer</summary>
>
>They share the same broad diffusion backbone for denoising crystal representations. What changes is the conditioning interface: DNG is open-ended and uses schedule steering, while CSP conditions on a target formula.
>
></details>


### Dataset note

Chemeleon-DNG provides checkpoints trained on both `mp-20` and `alex_mp_20`. In this notebook we explicitly use the `alex_mp_20` DNG and CSP checkpoints. For the open-ended DNG runs, however, the current repo only ships one built-in atom-count prior in `NUM_ATOM_DISTRIBUTIONS`, keyed as `mp-20`. So the DNG demos below use the `alex_mp_20` checkpoint together with the repo's available `mp-20` atom-count prior. The repo also notes that these checkpoints use a reduced `256`-step diffusion process for faster inference. So when you judge the generated samples, it is worth remembering that the prior still comes from a small-cell inorganic crystal dataset, not from the full text-guided `mp-40` setup described in the main Chemeleon repository.


## 3) DNG quickstart: generate crystals from scratch

We start Chemeleon-DNG with the purest task: de novo generation.

This first call still plays two roles:

- it checks that the repo-local Chemeleon-DNG environment is working,
- and it gives us a baseline DNG batch before we start steering the atom-count schedule in the next section.

### What to notice

- DNG does not need a target formula.
- The first call may still download checkpoints if `ckpts/` is empty.
- The notebook uses a reduced demo-length reverse process so this quickstart stays practical on CPU.
- After the quickstart batch, we can ask a more interesting question: how much can we move the output distribution by changing the atom-count schedule?


In [ ]:
from ase.io import read

CHEMELEON_DNG_CHECKPOINT_DATASET = globals().get('CHEMELEON_DNG_CHECKPOINT_DATASET', 'alex_mp_20')
CHEMELEON_DNG_PRIOR_NAME = globals().get('CHEMELEON_DNG_PRIOR_NAME', 'mp-20')
CHEMELEON_DNG_ALEX_MP20_MODEL_PATH = globals().get('CHEMELEON_DNG_ALEX_MP20_MODEL_PATH', 'ckpts/chemeleon_dng_alex_mp_20_v0.0.2.ckpt')

CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES = int(globals().get('CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES', 8))  # @param {type:"integer"}
CHEMELEON_DNG_QUICKSTART_BATCH_SIZE = int(globals().get('CHEMELEON_DNG_QUICKSTART_BATCH_SIZE', 8))  # @param {type:"integer"}
CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING = globals().get('CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING', False)  # @param {type:"boolean"}


def _chemeleon_quickstart_summary() -> str:
    lines = [
        'Chemeleon-DNG quickstart settings:',
        f'  num_samples: {CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES}',
        f'  batch_size: {CHEMELEON_DNG_QUICKSTART_BATCH_SIZE}',
        f'  atom-count prior: {CHEMELEON_DNG_PRIOR_NAME}',
        f'  checkpoint dataset: {CHEMELEON_DNG_CHECKPOINT_DATASET}',
        f'  checkpoint: {CHEMELEON_DNG_ALEX_MP20_MODEL_PATH}',
        f'  reuse_existing: {CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING}',
    ]
    if not IN_COLAB and widgets is not None:
        lines.append('Adjust the widgets, then rerun this cell to launch a new DNG quickstart batch.')
    return '\n'.join(lines)


def _apply_chemeleon_quickstart_controls(num_samples, batch_size, reuse_existing, announce: bool = True):
    global CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES, CHEMELEON_DNG_QUICKSTART_BATCH_SIZE, CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING
    CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES = int(num_samples)
    CHEMELEON_DNG_QUICKSTART_BATCH_SIZE = int(batch_size)
    CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING = bool(reuse_existing)
    summary = _chemeleon_quickstart_summary()
    if announce:
        print(summary)
    return summary


if IN_COLAB or widgets is None:
    print(_apply_chemeleon_quickstart_controls(CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES, CHEMELEON_DNG_QUICKSTART_BATCH_SIZE, CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING, announce=False))
    if not IN_COLAB and widgets is None:
        print('Install `ipywidgets` to get Jupyter controls for this cell.')
else:
    quickstart_num_samples_widget = widgets.BoundedIntText(value=CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES, min=1, max=24, description='Samples:', style={'description_width': '80px'}, layout=widgets.Layout(width='220px'))
    quickstart_batch_widget = widgets.BoundedIntText(value=CHEMELEON_DNG_QUICKSTART_BATCH_SIZE, min=1, max=24, description='Batch size:', style={'description_width': '80px'}, layout=widgets.Layout(width='220px'))
    quickstart_reuse_widget = widgets.Checkbox(value=bool(CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING), description='Reuse existing CIFs')
    quickstart_help = widgets.HTML('<small>In Jupyter, update the controls below and rerun this cell to resample the DNG quickstart batch.</small>')
    quickstart_status = widgets.HTML()
    display(widgets.VBox([widgets.HBox([quickstart_num_samples_widget, quickstart_batch_widget]), quickstart_reuse_widget, quickstart_help, quickstart_status]))

    def _refresh_chemeleon_quickstart(num_samples, batch_size, reuse_existing):
        quickstart_status.value = format_widget_pre(
            _apply_chemeleon_quickstart_controls(num_samples, batch_size, reuse_existing, announce=False)
        )

    bind_widget_state(
        {'num_samples': quickstart_num_samples_widget, 'batch_size': quickstart_batch_widget, 'reuse_existing': quickstart_reuse_widget},
        _refresh_chemeleon_quickstart,
    )

dng_output_dir = CHEMELEON_DNG_OUT / 'dng_quickstart'
dng_cif_paths = sample_chemeleon_dng(task='dng', num_samples=CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES, batch_size=CHEMELEON_DNG_QUICKSTART_BATCH_SIZE, output_dir=dng_output_dir, device=CHEMELEON_DNG_DEVICE, num_atom_distribution=CHEMELEON_DNG_PRIOR_NAME, model_path=CHEMELEON_DNG_ALEX_MP20_MODEL_PATH, reuse_existing=CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING)
dng_atoms_list = [read(path) for path in dng_cif_paths]
print('Generated:', len(dng_atoms_list), 'structures')
for i, atoms in enumerate(dng_atoms_list):
    print(i, atoms.get_chemical_formula(), len(atoms), f'{atoms.get_volume():.2f}')
show_atoms_gallery(dng_atoms_list, 'Chemeleon-DNG DNG quickstart', CHEMELEON_DNG_OUT / 'dng_quickstart_gallery.png')


**Task for you**
<div class="alert alert-block alert-info">

- Before the DNG steering plots appear, predict which statistic should respond most directly to the atom-count schedule.
- Before the CSP analysis, guess which is easier to verify from raw outputs: a scalar target or a formula target.
- Keep separating three ideas in your notes: `task`, `conditioning`, `screening`.

</div>


## 4) DNG steering and output analysis

Now we go beyond the default DNG call and use one of the real control knobs that Chemeleon-DNG exposes: the atom-count distribution.

We compare three DNG settings:

- the default `mp-20` atom-count prior shipped in the repo,
- a **small-cell** schedule,
- a **large-cell** schedule.

This is not property conditioning in the MatterGen sense, but it *is* a genuine sampling control. It is useful whenever you want to bias the search toward smaller or larger unit cells before doing any downstream screening.

This section also reuses the baseline DNG quickstart samples if they already exist on disk, so you can rerun the steering study without needing the earlier quickstart variable state to still be alive in memory.

### What to notice

- The atom-count schedule should move the `n_sites` distribution directly.
- That in turn often changes volume and density distributions.
- This is a good example of a practical generative control that is easy to explain and easy to verify.
- Just like in the MatterGen section, we can rank the low-density and high-density tails after generation to decide what to inspect next.

> **Quick check:** what is the model control you are actually changing in this DNG steering study?
>
><details><summary>Answer</summary>
>
>You are changing the distribution of the requested number of atoms per generated cell. That is a direct sampling control, and it should show up most clearly in the `n_sites` and volume statistics.
>
></details>


In [ ]:
from ase.io import read

CHEMELEON_DNG_CHECKPOINT_DATASET = globals().get('CHEMELEON_DNG_CHECKPOINT_DATASET', 'alex_mp_20')
CHEMELEON_DNG_PRIOR_NAME = globals().get('CHEMELEON_DNG_PRIOR_NAME', 'mp-20')
CHEMELEON_DNG_ALEX_MP20_MODEL_PATH = globals().get('CHEMELEON_DNG_ALEX_MP20_MODEL_PATH', 'ckpts/chemeleon_dng_alex_mp_20_v0.0.2.ckpt')

CHEMELEON_DNG_SMALL_SCHEDULE_TEXT = globals().get('CHEMELEON_DNG_SMALL_SCHEDULE_TEXT', '6, 6, 7')  # @param {type:"string"}
CHEMELEON_DNG_LARGE_SCHEDULE_TEXT = globals().get('CHEMELEON_DNG_LARGE_SCHEDULE_TEXT', '14, 14, 16')  # @param {type:"string"}
CHEMELEON_DNG_STEERING_REUSE_EXISTING = globals().get('CHEMELEON_DNG_STEERING_REUSE_EXISTING', True)  # @param {type:"boolean"}


def _parse_chemeleon_schedule(text: str, label: str):
    items = [item.strip() for item in str(text).split(',') if item.strip()]
    if not items:
        raise ValueError(f'{label} schedule cannot be empty.')
    schedule = [int(item) for item in items]
    if any(value <= 0 for value in schedule):
        raise ValueError(f'{label} schedule must contain positive integers only.')
    return schedule


def _chemeleon_steering_summary() -> str:
    lines = [
        'Chemeleon-DNG steering settings:',
        f'  small-cell schedule: {CHEMELEON_DNG_SMALL_SCHEDULE}',
        f'  large-cell schedule: {CHEMELEON_DNG_LARGE_SCHEDULE}',
        f'  baseline prior: {CHEMELEON_DNG_PRIOR_NAME}',
        f'  checkpoint dataset: {CHEMELEON_DNG_CHECKPOINT_DATASET}',
        f'  checkpoint: {CHEMELEON_DNG_ALEX_MP20_MODEL_PATH}',
        f'  reuse_existing: {CHEMELEON_DNG_STEERING_REUSE_EXISTING}',
    ]
    if not IN_COLAB and widgets is not None:
        lines.append('Adjust the schedules, then rerun this cell to regenerate the steering comparison.')
    return '\n'.join(lines)


def _apply_chemeleon_steering_controls(small_text, large_text, reuse_existing, announce: bool = True):
    global CHEMELEON_DNG_SMALL_SCHEDULE_TEXT, CHEMELEON_DNG_LARGE_SCHEDULE_TEXT
    global CHEMELEON_DNG_SMALL_SCHEDULE, CHEMELEON_DNG_LARGE_SCHEDULE, CHEMELEON_DNG_STEERING_REUSE_EXISTING
    CHEMELEON_DNG_SMALL_SCHEDULE_TEXT = str(small_text)
    CHEMELEON_DNG_LARGE_SCHEDULE_TEXT = str(large_text)
    CHEMELEON_DNG_SMALL_SCHEDULE = _parse_chemeleon_schedule(CHEMELEON_DNG_SMALL_SCHEDULE_TEXT, 'small-cell')
    CHEMELEON_DNG_LARGE_SCHEDULE = _parse_chemeleon_schedule(CHEMELEON_DNG_LARGE_SCHEDULE_TEXT, 'large-cell')
    CHEMELEON_DNG_STEERING_REUSE_EXISTING = bool(reuse_existing)
    summary = _chemeleon_steering_summary()
    if announce:
        print(summary)
    return summary


if IN_COLAB or widgets is None:
    print(_apply_chemeleon_steering_controls(CHEMELEON_DNG_SMALL_SCHEDULE_TEXT, CHEMELEON_DNG_LARGE_SCHEDULE_TEXT, CHEMELEON_DNG_STEERING_REUSE_EXISTING, announce=False))
    if not IN_COLAB and widgets is None:
        print('Install `ipywidgets` to get Jupyter controls for this cell.')
else:
    small_schedule_widget = widgets.Text(value=str(CHEMELEON_DNG_SMALL_SCHEDULE_TEXT), description='Small:', style={'description_width': '70px'}, layout=widgets.Layout(width='320px'))
    large_schedule_widget = widgets.Text(value=str(CHEMELEON_DNG_LARGE_SCHEDULE_TEXT), description='Large:', style={'description_width': '70px'}, layout=widgets.Layout(width='320px'))
    steering_reuse_widget = widgets.Checkbox(value=bool(CHEMELEON_DNG_STEERING_REUSE_EXISTING), description='Reuse existing CIFs')
    steering_help = widgets.HTML('<small>Use comma-separated atom counts. The schedule length sets how many samples are generated in each steering run.</small>')
    steering_status = widgets.HTML()
    display(widgets.VBox([widgets.HBox([small_schedule_widget, large_schedule_widget]), steering_reuse_widget, steering_help, steering_status]))

    def _refresh_chemeleon_steering(small_text, large_text, reuse_existing):
        steering_status.value = format_widget_pre(
            _apply_chemeleon_steering_controls(small_text, large_text, reuse_existing, announce=False)
        )

    bind_widget_state(
        {'small_text': small_schedule_widget, 'large_text': large_schedule_widget, 'reuse_existing': steering_reuse_widget},
        _refresh_chemeleon_steering,
    )

_apply_chemeleon_steering_controls(CHEMELEON_DNG_SMALL_SCHEDULE_TEXT, CHEMELEON_DNG_LARGE_SCHEDULE_TEXT, CHEMELEON_DNG_STEERING_REUSE_EXISTING, announce=False)
if 'dng_atoms_list' not in globals() or not dng_atoms_list:
    baseline_num_samples = int(globals().get('CHEMELEON_DNG_QUICKSTART_NUM_SAMPLES', 8))
    baseline_batch_size = int(globals().get('CHEMELEON_DNG_QUICKSTART_BATCH_SIZE', baseline_num_samples))
    baseline_reuse_existing = bool(globals().get('CHEMELEON_DNG_QUICKSTART_REUSE_EXISTING', True))
    baseline_output_dir = CHEMELEON_DNG_OUT / 'dng_quickstart'
    baseline_cif_paths = sample_chemeleon_dng(
        task='dng',
        num_samples=baseline_num_samples,
        batch_size=baseline_batch_size,
        output_dir=baseline_output_dir,
        device=CHEMELEON_DNG_DEVICE,
        num_atom_distribution=CHEMELEON_DNG_PRIOR_NAME,
        model_path=CHEMELEON_DNG_ALEX_MP20_MODEL_PATH,
        reuse_existing=baseline_reuse_existing,
    )
    dng_atoms_list = [read(path) for path in baseline_cif_paths]
    print(f'Loaded {len(dng_atoms_list)} baseline DNG structures for comparison')

dng_small_output_dir = CHEMELEON_DNG_OUT / 'dng_small_cells'
dng_large_output_dir = CHEMELEON_DNG_OUT / 'dng_large_cells'
dng_small_cif_paths = sample_chemeleon_dng(task='dng', num_samples=len(CHEMELEON_DNG_SMALL_SCHEDULE), batch_size=len(CHEMELEON_DNG_SMALL_SCHEDULE), output_dir=dng_small_output_dir, device=CHEMELEON_DNG_DEVICE, num_atom_distribution=CHEMELEON_DNG_SMALL_SCHEDULE, model_path=CHEMELEON_DNG_ALEX_MP20_MODEL_PATH, reuse_existing=CHEMELEON_DNG_STEERING_REUSE_EXISTING)
dng_large_cif_paths = sample_chemeleon_dng(task='dng', num_samples=len(CHEMELEON_DNG_LARGE_SCHEDULE), batch_size=len(CHEMELEON_DNG_LARGE_SCHEDULE), output_dir=dng_large_output_dir, device=CHEMELEON_DNG_DEVICE, num_atom_distribution=CHEMELEON_DNG_LARGE_SCHEDULE, model_path=CHEMELEON_DNG_ALEX_MP20_MODEL_PATH, reuse_existing=CHEMELEON_DNG_STEERING_REUSE_EXISTING)
dng_run_atoms = {'baseline_prior': dng_atoms_list, 'small_cells': [read(path) for path in dng_small_cif_paths], 'large_cells': [read(path) for path in dng_large_cif_paths]}
dng_run_titles = {'baseline_prior': 'Default mp-20 prior with alex checkpoint', 'small_cells': 'Small-cell schedule', 'large_cells': 'Large-cell schedule'}
dng_colors = {'baseline_prior': '#7aa2f7', 'small_cells': '#9ece6a', 'large_cells': '#f7768e'}
dng_analysis = analyze_dng_runs(dng_run_atoms, dng_run_titles, CHEMELEON_DNG_OUT, colors=dng_colors)
dng_rows = dng_analysis['rows']
dng_sorted_by_density = dng_analysis['sorted_by_density']


## 5) CSP quickstart: formula-conditioned generation

Now we switch from open-ended DNG to the explicitly conditioned Chemeleon-DNG task.

Here the condition is a target **formula**, so the chemistry is fixed and the model only has to infer plausible crystal structures for that composition.

### What to notice

- This is the cleanest Chemeleon-DNG example of conditional generation.
- Unlike MatterGen's scalar target, a formula condition is easy to verify directly from the output stoichiometry.
- The natural application is formula screening: pick one or two candidate chemistries, generate several structures for each, then rank them with cheap diagnostics before doing more expensive physics.
- We keep two formulas here so the notebook can show a real conditional comparison rather than a one-off gallery.

> **Quick check:** why is CSP a stronger demonstration of conditioning than a single DNG run?
>
><details><summary>Answer</summary>
>
>Because the condition is explicit and verifiable. You can directly check whether the generated structures respect the requested composition and whether different formulas occupy different structural regimes.
>
></details>


In [ ]:
from ase.io import read

CHEMELEON_CSP_TARGETS_TEXT = globals().get('CHEMELEON_CSP_TARGETS_TEXT', 'NaCl, LiMnO2')  # @param {type:"string"}
CSP_SAMPLES_PER_FORMULA = int(globals().get('CSP_SAMPLES_PER_FORMULA', 2))  # @param {type:"integer"}
CHEMELEON_CSP_BATCH_SIZE = int(globals().get('CHEMELEON_CSP_BATCH_SIZE', 2))  # @param {type:"integer"}
CHEMELEON_CSP_REUSE_EXISTING = globals().get('CHEMELEON_CSP_REUSE_EXISTING', True)  # @param {type:"boolean"}


def _parse_csp_targets(text: str):
    targets = [item.strip() for item in str(text).split(',') if item.strip()]
    if not targets:
        raise ValueError('Provide at least one CSP target formula.')
    return targets


def _chemeleon_csp_summary() -> str:
    lines = [
        'Chemeleon-DNG CSP settings:',
        f'  formulas: {csp_targets}',
        f'  samples_per_formula: {CSP_SAMPLES_PER_FORMULA}',
        f'  batch_size: {CHEMELEON_CSP_BATCH_SIZE}',
        f'  checkpoint: {CHEMELEON_CSP_ALEX_MP20_MODEL_PATH}',
        f'  reuse_existing: {CHEMELEON_CSP_REUSE_EXISTING}',
    ]
    if not IN_COLAB and widgets is not None:
        lines.append('Adjust the widgets, then rerun this cell to launch a new CSP comparison.')
    return '\n'.join(lines)


def _apply_chemeleon_csp_controls(targets_text, samples_per_formula, batch_size, reuse_existing, announce: bool = True):
    global CHEMELEON_CSP_TARGETS_TEXT, csp_targets, CSP_SAMPLES_PER_FORMULA
    global CHEMELEON_CSP_BATCH_SIZE, CHEMELEON_CSP_REUSE_EXISTING
    CHEMELEON_CSP_TARGETS_TEXT = str(targets_text)
    csp_targets = _parse_csp_targets(CHEMELEON_CSP_TARGETS_TEXT)
    CSP_SAMPLES_PER_FORMULA = int(samples_per_formula)
    CHEMELEON_CSP_BATCH_SIZE = int(batch_size)
    CHEMELEON_CSP_REUSE_EXISTING = bool(reuse_existing)
    summary = _chemeleon_csp_summary()
    if announce:
        print(summary)
    return summary


if IN_COLAB or widgets is None:
    print(_apply_chemeleon_csp_controls(CHEMELEON_CSP_TARGETS_TEXT, CSP_SAMPLES_PER_FORMULA, CHEMELEON_CSP_BATCH_SIZE, CHEMELEON_CSP_REUSE_EXISTING, announce=False))
    if not IN_COLAB and widgets is None:
        print('Install `ipywidgets` to get Jupyter controls for this cell.')
else:
    csp_targets_widget = widgets.Text(value=str(CHEMELEON_CSP_TARGETS_TEXT), description='Formulas:', style={'description_width': '70px'}, layout=widgets.Layout(width='360px'))
    csp_samples_widget = widgets.BoundedIntText(value=CSP_SAMPLES_PER_FORMULA, min=1, max=8, description='Samples:', style={'description_width': '70px'}, layout=widgets.Layout(width='200px'))
    csp_batch_widget = widgets.BoundedIntText(value=CHEMELEON_CSP_BATCH_SIZE, min=1, max=8, description='Batch size:', style={'description_width': '70px'}, layout=widgets.Layout(width='200px'))
    csp_reuse_widget = widgets.Checkbox(value=bool(CHEMELEON_CSP_REUSE_EXISTING), description='Reuse existing CIFs')
    csp_help = widgets.HTML('<small>Use comma-separated formulas. Update the controls below and rerun this cell to regenerate the CSP galleries.</small>')
    csp_status = widgets.HTML()
    display(widgets.VBox([widgets.HBox([csp_targets_widget, csp_samples_widget, csp_batch_widget]), csp_reuse_widget, csp_help, csp_status]))

    def _refresh_chemeleon_csp(targets_text, samples_per_formula, batch_size, reuse_existing):
        csp_status.value = format_widget_pre(
            _apply_chemeleon_csp_controls(targets_text, samples_per_formula, batch_size, reuse_existing, announce=False)
        )

    bind_widget_state(
        {'targets_text': csp_targets_widget, 'samples_per_formula': csp_samples_widget, 'batch_size': csp_batch_widget, 'reuse_existing': csp_reuse_widget},
        _refresh_chemeleon_csp,
    )

_apply_chemeleon_csp_controls(CHEMELEON_CSP_TARGETS_TEXT, CSP_SAMPLES_PER_FORMULA, CHEMELEON_CSP_BATCH_SIZE, CHEMELEON_CSP_REUSE_EXISTING, announce=False)
csp_samples = {}
for formula in csp_targets:
    out_dir = CHEMELEON_DNG_OUT / f'csp_{formula.lower()}'
    cif_paths = sample_chemeleon_dng(task='csp', formulas=[formula], num_samples=CSP_SAMPLES_PER_FORMULA, batch_size=CHEMELEON_CSP_BATCH_SIZE, output_dir=out_dir, device=CHEMELEON_DNG_DEVICE, model_path=CHEMELEON_CSP_ALEX_MP20_MODEL_PATH, reuse_existing=CHEMELEON_CSP_REUSE_EXISTING)
    atoms_list = [read(path) for path in cif_paths]
    csp_samples[formula] = atoms_list
    print(f'{formula}: {len(atoms_list)} structures')
    show_atoms_gallery(atoms_list, f'Chemeleon-DNG CSP: {formula}', CHEMELEON_DNG_OUT / f'csp_{formula.lower()}_gallery.png', subtitles=[f"{atoms.get_chemical_formula()} | {len(atoms)} atoms | volume={atoms.get_volume():.1f} Å³" for atoms in atoms_list])


## 6) CSP distributions, galleries, and screening

Once we have multiple CSP samples per formula, the notebook can do more than show one gallery. We can compare how the target chemistry changes density, volume, and atom-count distributions, then rank the outputs as a tiny formula-conditioned discovery workflow.

### What to notice

- Formula conditioning should separate the outputs into different structural regions.
- The same cheap screening ideas from the MatterGen section still apply here.
- Ranking the densest or lightest candidates is not a claim about thermodynamic stability; it is a practical way to decide which generated structures to inspect next.
- The formula-match table is useful because it separates “did the generator respect the chemistry?” from “which respected structures look most interesting?”

> **Quick check:** what is the conceptual difference between *conditioning* and *screening* in this CSP workflow?
>
><details><summary>Answer</summary>
>
>Conditioning tells the generator what chemistry to realize. Screening happens afterward, when you compute simple statistics such as density or volume to decide which generated candidates deserve closer study.
>
></details>


In [ ]:
csp_analysis = analyze_csp_samples(csp_samples, csp_targets, CHEMELEON_DNG_OUT)
csp_rows = csp_analysis["rows"]
csp_sorted_by_density = csp_analysis["sorted_by_density"]


## 7) When to choose Chemeleon-DNG


Chemeleon-DNG is easiest to understand once you line it up against the other Day 5 notebooks:

- Compared with `crystal-diffusion-from-scratch.ipynb`, the core diffusion idea is the same but the infrastructure is pretrained and task-oriented.
- Compared with `mattergen-crystals.ipynb`, the main user choice is not a scalar target but the task itself: open-ended DNG or formula-conditioned CSP.
- In both tasks, useful scientific work still happens after sampling, when you screen densities, atom counts, formulas, and candidate galleries.

If you can explain the decision table below, you can explain the practical difference between sampling control, explicit conditioning, and post-generation screening in modern crystal generation workflows.


### Decision table

| If your goal is... | Prefer | Why |
| --- | --- | --- |
| explore open-ended crystal generation with a lightweight control knob | Chemeleon-DNG DNG | the atom-count schedule is simple, visible, and easy to verify |
| generate candidates for a fixed formula and check adherence directly | Chemeleon-DNG CSP | the target chemistry is explicit and visible in the outputs |
| teach the cleanest direct continuation of the scratch crystal notebook | MatterGen | scalar-property steering is the closer conceptual bridge |
| compare candidate pools with the same cheap diagnostics | either | both workflows end in density / volume / atom-count screening and galleries |
| move quickly between open-ended discovery and formula-conditioned search | Chemeleon-DNG | the toolkit exposes both tasks through one user-facing interface |


## 8) Exercises

1. **DNG vs CSP:** In your own words, what is the difference between DNG and CSP?

<details><summary>Answer</summary>

DNG generates structures from scratch, while CSP generates structures conditioned on a target formula.

</details>

2. **Band gap vs formula conditioning:** How does MatterGen's low-vs-high band-gap sweep differ from Chemeleon-DNG's CSP sweep?

<details><summary>Answer</summary>

MatterGen changes a scalar target while leaving the chemistry open, whereas CSP fixes the chemistry directly and asks the model to generate plausible structures for that formula.

</details>

3. **Why density still matters:** Why did this notebook keep plotting density even when density was not always the conditioned target?

<details><summary>Answer</summary>

Because density is a cheap structural diagnostic and a useful screening statistic. It helps show whether different conditions move the generated structures into different structural regimes.

</details>

4. **Pick the right tool:** When would you choose Chemeleon-DNG over MatterGen?

<details><summary>Answer</summary>

Chemeleon-DNG is a good choice when you want a compact DNG/CSP workflow, especially if your main question is composition-conditioned structure generation.

</details>

5. **Screening workflow:** Suppose you wanted to prioritize compact, high-density candidates from either model. What would the next step be after generation?

<details><summary>Answer</summary>

Collect the generated structures, compute a screening statistic such as density or volume, rank the candidates, and then pass the most interesting ones to a more expensive downstream workflow such as relaxation or DFT.

</details>


## 9) Troubleshooting

- If `chemeleon_dng_repo/` exists but has no `pyproject.toml`, rerun the setup cell. The setup checks for incomplete clones, moves them aside, and reclones the repo cleanly.
- The Python environment lives in `tempfile.gettempdir()/chemeleon-dng-venv` rather than inside the repo. That avoids incomplete `torch` and `numpy` installs on mounted filesystems such as OneDrive and some WSL/Colab paths.
- The notebook anchors itself to the repo root, so outputs land inside `notebooks/05-generative/` for this tutorial repo rather than the parent directory.
- The first DNG or CSP run downloads about $523\,\mathrm{MB}$ of checkpoints into `chemeleon_dng_repo/ckpts/`. If that download is interrupted, rerun the first sampling cell.
- The Chemeleon subprocess forces `MPLBACKEND='Agg'` and uses a writable temporary Matplotlib cache, so it does not inherit Jupyter's inline backend and fail during import.
- The sampling helper reuses existing CIFs by default when the output directory already contains the expected number of structures. If you want a fully fresh run, delete the relevant output directory first.
- `CHEMELEON_DNG_DEMO_TIMESTEPS` controls the notebook-speed reverse process. Increase it if you want higher-fidelity samples and can tolerate longer runtimes.

## 10) Next steps

- In MatterGen, try a second conditioned checkpoint such as `chemical_system` or `space_group` and repeat the same sweep-plus-screening workflow.
- In Chemeleon-DNG, expand the CSP formula list and compare how the density-volume clouds move as the chemistry changes.
- Increase the DNG atom-count schedules from the tiny tutorial values here to broader small-cell and large-cell sweeps once you move to GPU.
- Take the screened high-density and low-density candidates from either model and pass them to a relaxation or stability-analysis workflow.
- Build one shared post-processing script that ingests MatterGen and Chemeleon outputs together, then ranks candidates by the same screening criteria.
